####1

In [ ]:
## install required packages
!pip install swig
!pip install wrds
!pip install pyportfolioopt
## install finrl library
!pip install git+https://github.com/AI4Finance-Foundation/FinRL.git

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 7.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 56.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 6.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 65.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 78.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.4/38.4 MB 11.3 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 24.1
    Uninstalling packaging-24.1:
      Successfully uninstalled packaging-24.1
  Attempting uninstall: numpy
    Found existing installation: numpy 1.25.2
    Uninstalling numpy-1.25.2:
      Successfully uninstalled numpy-1.25.2
  Attempting uninstall: scipy
    Found existing installation: scipy 1.11.4
    Uninstalling scipy-1.11.4:
      Successfully uninstalled scipy-1.11.4
  Attempting uninstall: pandas
    Found existing install

# Preprocess Data

In [ ]:
import pandas as pd
import numpy as np
import datetime
import yfinance as yf

from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.meta.preprocessor.preprocessors import FeatureEngineer, data_split
from finrl import config_tickers
from finrl.config import INDICATORS

import itertools

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
TRAIN_START_DATE = '2009-01-01'
TRAIN_END_DATE = '2022-01-01'
TRADE_START_DATE = '2022-01-02'
TRADE_END_DATE = '2024-06-01'
symbols = [
            'AAPL',
            'MSFT',
            'AMZN',
            'NVDA',
            'AMD'
            ]

df = YahooDownloader(start_date = TRAIN_START_DATE,
                                end_date = TRADE_END_DATE,
                                ticker_list = symbols).fetch_data()

df.head(2)

[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed


Shape of DataFrame:  (19395, 8)


,date,open,high,low,close,volume,tic,day
0,2009-01-02,3.067143,3.251429,3.041429,2.740172,746015200,AAPL,4
1,2009-01-02,2.190000,2.430000,2.170000,2.380000,13832100,AMD,4


In [ ]:
INDICATORS = ['macd',
               'rsi',
               'cci',
               'dx']

fe = FeatureEngineer(use_technical_indicator=True,
                     tech_indicator_list = INDICATORS,
                     use_vix=False,
                     use_turbulence=True,
                     user_defined_feature = False)

processed = fe.preprocess_data(df)

processed = processed.drop(columns=['open','high','low','volume','day'])

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


Successfully added technical indicators
Successfully added turbulence index


In [ ]:
list_ticker = processed["tic"].unique().tolist()
list_date = list(pd.date_range(processed['date'].min(),processed['date'].max()).astype(str))
combination = list(itertools.product(list_date,list_ticker))

processed_full = pd.DataFrame(combination,columns=["date","tic"]).merge(processed,on=["date","tic"],how="left")
processed_full = processed_full[processed_full['date'].isin(processed['date'])]
processed_full = processed_full.sort_values(['date','tic'])

processed_full = processed_full.fillna(0)

,date,tic,close,macd,rsi,cci,dx,turbulence
0,2009-01-02,AAPL,2.740172,0.000000,100.000000,66.666667,100.000000,0.000000
1,2009-01-02,AMD,2.380000,0.000000,100.000000,66.666667,100.000000,0.000000
2,2009-01-02,AMZN,2.718000,0.000000,100.000000,66.666667,100.000000,0.000000
3,2009-01-02,MSFT,15.011640,0.000000,100.000000,66.666667,100.000000,0.000000
4,2009-01-02,NVDA,0.199726,0.000000,100.000000,66.666667,100.000000,0.000000
...,...,...,...,...,...,...,...,...
28140,2024-05-31,AAPL,192.250000,4.254256,67.294615,83.902344,18.866838,2.059761
28141,2024-05-31,AMD,166.899994,1.943916,55.488332,38.653476,4.728638,2.059761
28142,2024-05-31,AMZN,176.440002,-0.715091,36.745153,-201.650314,46.940153,2.059761
28143,2024-05-31,MSFT,415.130005,3.323865,46.929727,-134.029450,50.967552,2.059761


In [ ]:
# Split the data
train = data_split(processed_full, TRAIN_START_DATE,TRAIN_END_DATE)
trade = data_split(processed_full, TRADE_START_DATE,TRADE_END_DATE)
print(len(train))
print(len(trade))

trade_path= '/content/drive/MyDrive/DRLP/trade.csv'
train_path = '/content/drive/MyDrive/DRLP/train.csv'

with open(train_path, 'w', encoding='utf-8-sig') as f:
    train.to_csv(f)

with open(trade_path, 'w', encoding = 'utf-8-sig') as f:
  trade.to_csv(f)

16365
3030


# Load the Data from and train the model

#### install RecurrentPPO

In [ ]:
!git clone https://github.com/DLR-RM/rl-baselines3-zoo
%cd rl-baselines3-zoo
!pip install -e .
"""!apt-get install swig cmake ffmpeg
!pip install -r requirements.txt
!pip install -e .[plots,tests]"""

fatal: destination path 'rl-baselines3-zoo' already exists and is not an empty directory.
/content/rl-baselines3-zoo
Obtaining file:///content/rl-baselines3-zoo
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.1/380.1 kB 12.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.1/111.1 kB 10.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.0/233.0 kB 23.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 9.3 MB/s eta 0:00:00
  Building editable for rl_zoo3 (pyproject.toml) ... done
  Created wheel for rl_zoo3: filename=rl_zoo3-2.4.0a4-0.editable-py3-none-any.whl size=4368 sha256=c229c60a389d7965cb90197343acd1bbe78fa3dd8587c1cce80e7f29656beff3


'!apt-get install swig cmake ffmpeg\n!pip install -r requirements.txt\n!pip install -e .[plots,tests]'

## Setup Environment and model

#### imports

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from sb3_contrib import RecurrentPPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.evaluation import evaluate_policy
from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv
from finrl.agents.stablebaselines3.models import DRLAgent
from finrl import config_tickers
from finrl.main import check_and_make_directories
from finrl.config import INDICATORS, TRAINED_MODEL_DIR, RESULTS_DIR
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#### env

In [ ]:

# Directory to save trained models and evaluation results
model_dir = "/content/drive/MyDrive/DRLP/Result"
check_and_make_directories([model_dir])

# Load the train and trade datasets
train = pd.read_csv('/content/drive/MyDrive/DRLP/train.csv')
train = train.set_index(train.columns[0])
train.index.names = ['']

trade = pd.read_csv('/content/drive/MyDrive/DRLP/trade.csv')
trade = trade.set_index(trade.columns[0])
trade.index.names = ['']

INDICATORS = ['macd', 'rsi', 'cci', 'dx']
stock_dimension = len(train.tic.unique())
state_space = 1 + 2 * stock_dimension + len(INDICATORS) * stock_dimension

# Extract turbulence values from the train dataset
turbulence_data = train[['date', 'turbulence']].drop_duplicates()
turbulence_values = turbulence_data['turbulence'].values

# Calculate the 90th percentile threshold
turbulence_threshold = np.percentile(turbulence_values, 90)

buy_cost_list = sell_cost_list = [0.001] * stock_dimension
num_stock_shares = [0] * stock_dimension

env_kwargs = {
    "hmax": 100,
    "initial_amount": 1000000,
    "num_stock_shares": num_stock_shares,
    "buy_cost_pct": buy_cost_list,
    "sell_cost_pct": sell_cost_list,
    "state_space": state_space,
    "stock_dim": stock_dimension,
    "tech_indicator_list": INDICATORS,
    "action_space": stock_dimension,
    "reward_scaling": 1e-4,
    "turbulence_threshold": turbulence_threshold
}

from gymnasium import Wrapper
from collections import deque


from gymnasium.spaces import Box
import gymnasium as gym

class StockTradingEnvWrapper(Wrapper):
    def __init__(self, env, window_size=10):
        super(StockTradingEnvWrapper, self).__init__(env)
        self.window_size = window_size
        self.observation_window = deque(maxlen=window_size)
        self.portfolio_values = []

        # Define the observation space to match the windowed observations
        single_obs_shape = env.observation_space.shape
        self.observation_space = Box(
            low=np.repeat(env.observation_space.low[np.newaxis, :], window_size, axis=0),
            high=np.repeat(env.observation_space.high[np.newaxis, :], window_size, axis=0),
            dtype=env.observation_space.dtype
        )

    def step(self, action):
        state, reward, done, truncated, info = self.env.step(action)
        state = state[0]  # Only use the list of values from the state tuple

        self.observation_window.append(state)
        if len(self.observation_window) < self.window_size:
            padded_state = [self.observation_window[0]] * (self.window_size - len(self.observation_window)) + list(self.observation_window)
        else:
            padded_state = list(self.observation_window)

        portfolio_value = self._calculate_portfolio_value()
        self.portfolio_values.append(portfolio_value)

        if done:
            info['portfolio_value'] = portfolio_value
            info.update(self._calculate_metrics())

        return np.array(padded_state), reward, done, truncated, info

    def reset(self, **kwargs):
        self.portfolio_values = []
        state = self.env.reset(**kwargs)
        state = state[0]  # Only use the list of values from the state tuple

        self.observation_window = deque([state] * self.window_size, maxlen=self.window_size)
        for idx, obs in enumerate(self.observation_window):
            print(f"Initial observation shape at index {idx}: {np.array(obs).shape}")

        return np.array(self.observation_window)

    def _calculate_portfolio_value(self):
        cash = self.env.state[0]
        stocks = np.array(self.env.state[1:self.env.stock_dim + 1])
        holdings = np.array(self.env.state[self.env.stock_dim + 1:2 * self.env.stock_dim + 1])
        return cash + np.sum(stocks * holdings)

    def _calculate_metrics(self):
        if not self.portfolio_values:
            return {}

        portfolio_series = pd.Series(self.portfolio_values)
        daily_returns = portfolio_series.pct_change().dropna()

        sharpe_ratio = np.nan
        if daily_returns.std() != 0:
            sharpe_ratio = (252 ** 0.5) * daily_returns.mean() / daily_returns.std()

        profits = daily_returns[daily_returns > 0]
        losses = daily_returns[daily_returns < 0]

        appt = (profits.sum() + losses.sum()) / len(daily_returns) if len(daily_returns) > 0 else np.nan
        mpb = np.max(portfolio_series)
        mer = daily_returns.mean() - 0.03 / 252
        cr = (portfolio_series.iloc[-1] / portfolio_series.iloc[0]) - 1

        return {
            'Sharpe Ratio': sharpe_ratio,
            'APPT': appt,
            'MPB': mpb,
            'MER': mer,
            'CR': cr
        }

class LSTMFeatureExtractor(BaseFeaturesExtractor):
    def __init__(self, observation_space, hidden_size=128, num_layers=1):
        super(LSTMFeatureExtractor, self).__init__(observation_space, features_dim=hidden_size)
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.lstm = None  # Initialize LSTM in the forward method
        self.linear = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size)
        )

    def forward(self, x):
       # print(f"Input shape: {x.shape}")

        # Add sequence length dimension if missing
        if len(x.shape) == 2:
            x = x.unsqueeze(1)  # Add sequence dimension

        if self.lstm is None:
            # Infer input size from the first observation
            input_size = x.shape[2]
            self.lstm = nn.LSTM(input_size=input_size,
                                hidden_size=self.hidden_size,
                                num_layers=self.num_layers,
                                batch_first=True).to(x.device)

        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)

        out, (hn, cn) = self.lstm(x, (h0, c0))
        features = hn[-1]  # Take the last hidden state
        return self.linear(features)

policy_kwargs = dict(
    features_extractor_class=LSTMFeatureExtractor,
    features_extractor_kwargs=dict(hidden_size=128, num_layers=1),
)

window_sizes = [5, 15, 30, 50]
results = {}




#### Training loop

In [ ]:

for window_size in window_sizes:
    e_train_gym = StockTradingEnv(df=train, **env_kwargs)
    e_train_env = StockTradingEnvWrapper(e_train_gym, window_size=window_size)

    env_train, _ = e_train_gym.get_sb_env()

    model = RecurrentPPO("MlpLstmPolicy", env_train, policy_kwargs=policy_kwargs, verbose=1)
    model.learn(total_timesteps=5000)

    vec_env = model.get_env()
    mean_reward, std_reward = evaluate_policy(model, vec_env, n_eval_episodes=3, warn=False)
    print(mean_reward)
    # Save the model
    model_path = os.path.join(model_dir, f"ppo_recurrent_window_{window_size}")
    model.save(model_path)
    del model  # Free memory


Using cpu device
----------------------------
| time/              |     |
|    fps             | 117 |
|    iterations      | 1   |
|    time_elapsed    | 1   |
|    total_timesteps | 128 |
----------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 31          |
|    iterations           | 2           |
|    time_elapsed         | 8           |
|    total_timesteps      | 256         |
| train/                  |             |
|    approx_kl            | 0.005729061 |
|    clip_fraction        | 0.00781     |
|    clip_range           | 0.2         |
|    entropy_loss         | -7.1        |
|    explained_variance   | -0.0619     |
|    learning_rate        | 0.0003      |
|    loss                 | -0.0101     |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.00866    |
|    std                  | 1           |
|    value_loss           | 0.0388      |
------------------------

# Evaluation loop

In [ ]:

for window_size in window_sizes:
    print(f"Evaluating model with window size {window_size}")

    # Reload the model
    model_path = os.path.join(model_dir, f"ppo_recurrent_window_{window_size}")
    model = RecurrentPPO.load(model_path)

    # Evaluate on trade dataset
    e_trade_gym = StockTradingEnv(df=trade, **env_kwargs)
    e_trade_env = StockTradingEnvWrapper(e_trade_gym, window_size=window_size)
    e_trade_env, _ = e_trade_env.get_sb_env()


    # Reset the environment
    obs = e_trade_env.reset()
    obs = np.array([obs])  # Ensure the observation is properly batched
    done = False

    portfolio_values = []

    while not done:
        action, _states = model.predict(obs, deterministic=True)
        action = action[0]  # Ensure the action is a 1-dimensional array
        obs, reward, done, truncated, info = e_trade_env.step(action)
        if isinstance(obs, tuple):
            obs = obs[0]  # Extract the list part of the observation
        obs = np.array([obs])  # Ensure the observation is properly batched
        portfolio_values.append(e_trade_env._calculate_portfolio_value())

    results[window_size] = portfolio_values

# Plot the results
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 6))

for window_size, values in results.items():
    dates = trade.index[:len(values)]
    plt.plot(dates, values, label=f'TimeWindow = {window_size}')

plt.xlabel('Date')
plt.ylabel('Return')
plt.legend()
plt.title('Portfolio Returns for Different Time Windows')
plt.show()

Evaluating model with window size 5


RuntimeError: Error(s) in loading state_dict for RecurrentActorCriticPolicy:
	Unexpected key(s) in state_dict: "features_extractor.lstm.weight_ih_l0", "features_extractor.lstm.weight_hh_l0", "features_extractor.lstm.bias_ih_l0", "features_extractor.lstm.bias_hh_l0", "pi_features_extractor.lstm.weight_ih_l0", "pi_features_extractor.lstm.weight_hh_l0", "pi_features_extractor.lstm.bias_ih_l0", "pi_features_extractor.lstm.bias_hh_l0", "vf_features_extractor.lstm.weight_ih_l0", "vf_features_extractor.lstm.weight_hh_l0", "vf_features_extractor.lstm.bias_ih_l0", "vf_features_extractor.lstm.bias_hh_l0". 